# German Industry Run-Ups, Crashes, and Post-Run-Up Performance

This notebook presents the German empirical analysis based on the framework of Greenwood, Shleifer, and You (2019), *Bubbles for Fama*. It is a Germany-focused adaptation, not a claim of an identical replication of the original U.S. sample.

The calculations were executed in Stata using `Untitled_revised`. This notebook documents the complete workflow, displays the executed exhibits, and interprets the German results. It does not recalculate the raw-data pipeline in Python.

## 1. Research question and empirical objectives

The main question is whether unusually large price increases in German industries are followed by elevated crash risk and distinctive post-run-up performance.

The analysis has five objectives:

1. Describe German industry episodes that satisfy a large prior price increase.
2. Examine subsequent returns and maximum drawdowns.
3. Compare crash probabilities after strict 100% run-ups with broader observations.
4. Study whether the run-up indicator predicts crashes in simple regressions, with and without a volatility control.
5. Visualize industry and market performance around the identified 100% run-up episodes.

The German results are interpreted against the broad patterns reported in the U.S. and international evidence, while recognizing that the German sample is smaller and has different industry composition and data coverage.

## 2. Input data and files that are not used

The Stata workflow uses `comp_global_daily.dta` as the source dataset. It filters this file to German common-equity securities and uses the security-level variables needed to construct industry portfolios.

Files such as `funda.dta`, `msci_rf.dta`, and `gdp.dta` are not read by the executed workflow. They are therefore not part of the calculations reported here. No risk-free return series is introduced, so risk-free-return fields are not reported as if they were observed.

The final exhibits are generated in Stata and read below from Excel or PNG files.

## 3. Portable project setup

The notebook searches the current directory, nearby parent directories, and their immediate child folders for the final exhibit files. This avoids hard-coding the local Windows path used by Stata.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

FINAL_FILES = [
    'germany_table1_paper_style.xlsx',
    'germany_event_time_paths.png',
    'germany_event_time_paths.xlsx',
    'germany_table3_main.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_crash_test.xlsx',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    for base in [cwd, *cwd.parents]:
        try:
            candidates.extend(item for item in base.iterdir() if item.is_dir())
        except OSError:
            pass
    scored = []
    for candidate in dict.fromkeys(candidates):
        if candidate.is_dir():
            try:
                present = sum((candidate / filename).exists() for filename in FINAL_FILES)
            except (OSError, PermissionError):
                continue
            if present > 0:
                scored.append((present, candidate))
    if not scored:
        raise FileNotFoundError('Could not find the folder containing the German exhibit files.')
    return sorted(scored, key=lambda item: (-item[0], str(item[1])))[0][1]

PROJECT_DIR = find_project_dir()
display(Markdown(f'**Project folder:** `{PROJECT_DIR}`'))
for filename in FINAL_FILES:
    print(f'{filename}: {"FOUND" if (PROJECT_DIR / filename).exists() else "MISSING"}')

def read_excel(filename):
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Missing required exhibit: {filename}')
    return pd.read_excel(path)


## 4. Security sample and currency normalization

The Stata workflow keeps observations with `fic == "DEU"`, keeps common equity using `tpci == "0"`, requires a valid `gsector` classification, and requires a positive price. Security identifiers are represented by `gvkey` and `iid`.

The adjusted total-return price is constructed as:

`(prccd × fx_eur / ajexdi) × trfd`

EUR observations use a conversion factor of one. DEM observations use the fixed conversion factor `1 / 1.95583`. This step is important for Germany because an unhandled currency transition could appear as a false market-wide price movement. The revised Stata workflow records currency changes as a diagnostic.

## 5. Daily return cleaning and monthly security returns

Daily returns are constructed from adjusted total-return prices. The workflow flags likely reversal errors when a very large current or previous return is followed by a small two-day compounded return. It also flags extreme current returns above 200%. Flagged daily returns are set to missing rather than being treated as genuine economic movements.

For monthly returns, the last available trading observation for each security-month is retained. A monthly return is calculated only when the previous security observation is the immediately preceding month. This prevents long gaps from being treated as ordinary one-month returns.

## 6. Industry portfolio construction

The unit of analysis is the industry-month. This is essential because the research question concerns industry run-ups rather than a single German market index.

For each industry-month, the workflow computes beginning-of-month market capitalization from price, the EUR conversion factor, and shares outstanding. Industry returns are value-weighted using lagged market capitalization. A German value-weighted market portfolio is constructed from the same securities.

The net-market industry return is defined as the industry return minus the German market return. Both raw and net-market returns are used in the run-up definition.

## 7. Run-up identification

For each threshold X in 50%, 75%, 100%, 125%, and 150%, an industry qualifies when all three conditions hold:

1. Prior 24-month raw industry return is at least X.
2. Prior 24-month net-market industry return is at least X.
3. Prior 60-month raw industry return is at least 50%.

The qualifying measures are lagged one month before the event indicator is assigned. The workflow then retains distinct episodes rather than counting every qualifying month. A qualifying observation is retained only when no qualifying observation appears in the preceding 24 months. Consequently, the episode count is an event count, not a month count.

## 8. Forward outcomes, drawdowns, and crashes

The outcome window covers the 24 months after each event. An observation is considered complete only when the required monthly sequence exists through month +24.

For each future month, the workflow compares the industry wealth index with the highest value observed from event month 0 through the month before the observation being evaluated. The maximum drawdown is the most negative peak-to-trough decline over the window. A raw or net-market crash indicator equals one when the relevant maximum drawdown is at least 40% in absolute value.

This definition differs from a simple end-to-end return: an industry can finish the two-year window above its starting value and still experience a crash if it falls by 40% from an earlier peak.

## 9. Pre-run-up volatility and the regression sample

Pre-run-up annualized volatility is calculated from value-weighted daily industry returns over the preceding 12 months and annualized using the square root of 252.

The full outcome file is used for the threshold summaries and formal crash-probability comparison. Volatility is merged only for the regression specifications that require it. This prevents missing volatility from unnecessarily changing the unconditional crash-test sample.

## 10. Descriptive German run-up episodes

This exhibit lists the identified raw-only 100% German run-up episodes and summarizes their subsequent returns and drawdowns. It provides the episode-level context for the aggregate threshold results below.

The table is descriptive. It does not by itself establish that run-ups cause crashes. The main interpretive focus is whether large subsequent drawdowns coexist with strong post-run-up average returns, since these measure different features of the path.

In [ ]:
table1 = read_excel('germany_table1_paper_style.xlsx')
display(table1)
print(f'Rows: {len(table1)} | Columns: {len(table1.columns)}')

interp = []
interp.append('### Interpretation')

if 'industry_name' in table1.columns:
    interp.append(f"- The descriptive episode table contains **{len(table1)} rows** across **{table1['industry_name'].nunique()} industries**.")
else:
    interp.append(f"- The descriptive episode table contains **{len(table1)} rows**.")

if 'outcome' in table1.columns:
    outcome_counts = table1['outcome'].value_counts(dropna=False)
    pieces = [f"{k}: {int(v)}" for k, v in outcome_counts.items()]
    interp.append('- Outcome split in this table: ' + ', '.join(pieces) + '.')

nrf_cols = [c for c in table1.columns if c.lower().startswith('nrf')]
if nrf_cols:
    all_missing = table1[nrf_cols].isna().all().all()
    if all_missing:
        interp.append('- The `nrf*` columns are `NaN` throughout. This is expected in the current workflow because no separate risk-free series is used; these fields are therefore unavailable rather than erroneous.')
    else:
        interp.append('- The `nrf*` columns are partially populated; interpret them only where non-missing values are present.')

if {'raw12_pct', 'raw24_pct'}.issubset(table1.columns):
    interp.append(f"- Mean forward raw returns in this strict sample are {table1['raw12_pct'].mean():.1f}% (12M) and {table1['raw24_pct'].mean():.1f}% (24M), showing that crash risk and average returns can move differently.")

if 'dd24_pct' in table1.columns:
    interp.append(f"- The average 24M drawdown is {table1['dd24_pct'].mean():.1f}%, highlighting substantial downside episodes even when some forward returns remain positive.")

display(Markdown('\n'.join(interp)))


## 11. Event-time price paths

The event-time analysis follows strict 100% run-up episodes from month −24 to month +48. Each episode is normalized at event month 0. The lines distinguish the average industry path, episodes later classified as crashes, episodes not classified as crashes, and the market path.

The relevant evidence is whether crash episodes separate from non-crash episodes after the event and whether the industry path differs from the market benchmark. Because the number of strict German episodes is small, the figure is best treated as visual and descriptive evidence.

In [ ]:
event_time = read_excel('germany_event_time_paths.xlsx')
display(event_time)
print(f'Event-time rows: {len(event_time)}')
if 'event_month' in event_time.columns:
    print(f'Rows at event month 0: {(event_time["event_month"] == 0).sum()}')
display(Image(filename=str(PROJECT_DIR / 'germany_event_time_paths.png')))

interp = []
interp.append('### Interpretation')
if 'event_month' in event_time.columns:
    interp.append(f"- The event-time panel has **{len(event_time)} rows**, spanning months {int(event_time['event_month'].min())} to {int(event_time['event_month'].max())}, with {(event_time['event_month'] == 0).sum()} row(s) at month 0.")

if {'event_month', 'crashes', 'noncrashes'}.issubset(event_time.columns):
    post = event_time[event_time['event_month'] > 0].copy()
    if len(post) > 0:
        below_count = int((post['crashes'] < post['noncrashes']).sum())
        interp.append(f"- After the event (months > 0), the crash path is below the non-crash path in **{below_count} of {len(post)} months**, consistent with weaker post-event trajectories for crash episodes.")

if {'event_month', 'average', 'market'}.issubset(event_time.columns):
    post = event_time[event_time['event_month'] > 0].copy()
    if len(post) > 0:
        market_below = int((post['market'] < post['average']).sum())
        interp.append(f"- The market path is below the average industry path in **{market_below} of {len(post)} post-event months** in the current output.")

strict_n = None
try:
    detail = read_excel('germany_crash_results.xlsx')
    if {'Pick-up threshold (%)', 'Number of run-ups identified'}.issubset(detail.columns):
        row100 = detail.loc[detail['Pick-up threshold (%)'] == 100, 'Number of run-ups identified']
        if not row100.empty:
            strict_n = int(row100.iloc[0])
except Exception:
    pass

if strict_n is not None:
    interp.append(f"- Interpretation should remain cautious because the strict 100% threshold is based on only **{strict_n} episodes**.")
else:
    interp.append('- Interpretation should remain cautious because the strict episode sample is small.')

display(Markdown('\n'.join(interp)))


## 12. Threshold-level crash and return results

The threshold exhibit summarizes the number of distinct episodes, forward raw returns, raw and net-market crash rates, and average 24-month drawdowns at each threshold.

The interpretation should look for a systematic pattern across thresholds, not just the largest percentage. Higher thresholds automatically produce fewer observations, so a high crash rate based on only a few episodes is imprecise. The German results should therefore be reported with both magnitudes and episode counts.

In [ ]:
table3 = read_excel('germany_table3_main.xlsx')
display(table3)
supporting_path = PROJECT_DIR / 'germany_crash_results.xlsx'
detailed = None
if supporting_path.exists():
    detailed = pd.read_excel(supporting_path)
    display(Markdown('### Detailed threshold output'))
    display(detailed)

interp = []
interp.append('### Interpretation')

rf_cols = [c for c in table3.columns if 'risk-free' in c.lower() or c.lower().startswith('nrf')]
if rf_cols:
    rf_all_missing = table3[rf_cols].isna().all().all()
    if rf_all_missing:
        interp.append('- In `germany_table3_main.xlsx`, the risk-free-return fields are `NaN` throughout. This is expected because the current Germany workflow does not include an external risk-free series; those columns should be read as unavailable fields, not failed calculations.')

if detailed is not None and {'Pick-up threshold (%)', 'Number of run-ups identified'}.issubset(detailed.columns):
    dd = detailed.sort_values('Pick-up threshold (%)').copy()
    thresholds = [int(x) for x in dd['Pick-up threshold (%)'].tolist()]
    counts = [int(x) for x in dd['Number of run-ups identified'].tolist()]
    interp.append('- Episode counts fall quickly as thresholds rise: ' + ', '.join(f"{t}%: {n}" for t, n in zip(thresholds, counts)) + '.')

    if 'Raw crashes (%)' in dd.columns:
        raw_rates = dd['Raw crashes (%)'].tolist()
        interp.append('- Raw crash rates by threshold are: ' + ', '.join(f"{t}%: {r:.1f}%" for t, r in zip(thresholds, raw_rates)) + '.')

    if 'Net-market crashes (%)' in dd.columns:
        net_rates = dd['Net-market crashes (%)'].tolist()
        interp.append('- Net-of-market crash rates by threshold are: ' + ', '.join(f"{t}%: {r:.1f}%" for t, r in zip(thresholds, net_rates)) + '.')

    if '24M drawdown (%)' in dd.columns:
        drawdowns = dd['24M drawdown (%)'].tolist()
        interp.append('- Average 24M raw drawdowns are large at every threshold: ' + ', '.join(f"{t}%: {v:.1f}%" for t, v in zip(thresholds, drawdowns)) + '.')

    if {'12M raw return (%)', '24M raw return (%)'}.issubset(dd.columns):
        r12 = dd['12M raw return (%)'].tolist()
        r24 = dd['24M raw return (%)'].tolist()
        interp.append('- Mean forward raw returns vary across thresholds (12M: ' + ', '.join(f"{t}%: {v:.1f}%" for t, v in zip(thresholds, r12)) + '; 24M: ' + ', '.join(f"{t}%: {v:.1f}%" for t, v in zip(thresholds, r24)) + ').')

    interp.append('- Because several thresholds rely on very few episodes, the pattern should not be presented as mechanically monotonic; magnitudes and sample size must be interpreted together.')
else:
    interp.append('- Use the detailed threshold file (`germany_crash_results.xlsx`) for substantive interpretation of episode counts, crash rates, drawdowns, and forward returns.')

display(Markdown('\n'.join(interp)))


## 13. Regression evidence

The regression table estimates crash indicators on raw and net-market run-up indicators at each threshold. The uncontrolled specification contains the run-up indicator alone. The controlled specification adds pre-run-up annualized volatility. Standard errors are clustered by calendar year.

The run-up coefficient is a conditional difference in crash probability within the specified linear-probability model. If the coefficient is smaller after adding volatility, that is consistent with volatility explaining part of the raw association. The small German episode sample means that coefficient size, p-values, and robustness across specifications should be considered together.

In [ ]:
regressions = read_excel('germany_footnote15_regressions.xlsx')
display(regressions)
print(f'Regression rows: {len(regressions)}')

interp = []
interp.append('### Interpretation')

if {'Volatility control', 'Observations'}.issubset(regressions.columns):
    obs_no = regressions.loc[regressions['Volatility control'].astype(str).str.lower() == 'no', 'Observations']
    obs_yes = regressions.loc[regressions['Volatility control'].astype(str).str.lower() == 'yes', 'Observations']
    if not obs_no.empty and not obs_yes.empty:
        interp.append(f"- The regressions use {int(obs_no.iloc[0])} observations without volatility and {int(obs_yes.iloc[0])} with volatility, reflecting a small row loss when volatility is required.")

if {'Return type', 'Threshold (%)', 'Volatility control', 'Run-up coefficient'}.issubset(regressions.columns):
    for rtype in ['raw', 'net']:
        subset = regressions[regressions['Return type'].astype(str).str.lower() == rtype].copy()
        if subset.empty:
            continue
        subset = subset.sort_values('Threshold (%)')
        no = subset[subset['Volatility control'].astype(str).str.lower() == 'no']
        yes = subset[subset['Volatility control'].astype(str).str.lower() == 'yes']
        if not no.empty:
            interp.append(f"- {rtype.capitalize()} run-up coefficients without volatility controls: " + ', '.join(f"{int(t)}%: {c:.4f}" for t, c in zip(no['Threshold (%)'], no['Run-up coefficient'])) + '.')
        if not yes.empty:
            interp.append(f"- {rtype.capitalize()} run-up coefficients with volatility controls: " + ', '.join(f"{int(t)}%: {c:.4f}" for t, c in zip(yes['Threshold (%)'], yes['Run-up coefficient'])) + '.')
        if not no.empty and not yes.empty:
            merged = no[['Threshold (%)', 'Run-up coefficient']].merge(
                yes[['Threshold (%)', 'Run-up coefficient']],
                on='Threshold (%)', suffixes=('_no', '_yes')
            )
            attenuated = (merged['Run-up coefficient_yes'] < merged['Run-up coefficient_no']).sum()
            interp.append(f"- For {rtype} run-ups, adding volatility reduces the coefficient at {int(attenuated)} of {len(merged)} thresholds, indicating attenuation but not elimination of the association.")

if 'Run-up p-value' in regressions.columns:
    share_sig = (regressions['Run-up p-value'] < 0.05).mean() * 100
    interp.append(f"- {share_sig:.0f}% of reported run-up p-values are below 0.05 in the current table.")

interp.append('- Coefficients are linear-probability differences in crash risk (percentage-point terms in probability units), not multiplicative risk ratios.')
interp.append('- Interpretation should remain cautious because strict episodes are few and standard errors are clustered only by calendar year.')

display(Markdown('\n'.join(interp)))


## 14. Formal crash-probability comparison

The formal test compares the crash probability in the full complete-outcome sample with the crash probability after a strict 100% run-up. It also compares the strict 100% group with non-100% months. The table reports the estimated probabilities, differences, z statistics, and p-values.

A large numerical difference with a large p-value should be described as suggestive rather than statistically conclusive. This distinction is especially important for Germany because the strict run-up group contains few episodes.

In [ ]:
crash_test = read_excel('germany_crash_test.xlsx')
display(crash_test)
if {'Comparison', 'Observations', 'Crashes'}.issubset(crash_test.columns):
    check = crash_test[['Comparison', 'Observations', 'Crashes']].copy()
    check['Calculated probability (%)'] = check['Crashes'] / check['Observations'] * 100
    display(check)

interp = []
interp.append('### Interpretation')

if {'Comparison', 'Observations', 'Crashes', 'Crash probability (%)'}.issubset(crash_test.columns):
    data = crash_test.set_index('Comparison')
    if 'Unconditional' in data.index and '100% run-up' in data.index:
        uncond_obs = int(data.loc['Unconditional', 'Observations'])
        uncond_cr = int(data.loc['Unconditional', 'Crashes'])
        uncond_p = float(data.loc['Unconditional', 'Crash probability (%)'])
        strict_obs = int(data.loc['100% run-up', 'Observations'])
        strict_cr = int(data.loc['100% run-up', 'Crashes'])
        strict_p = float(data.loc['100% run-up', 'Crash probability (%)'])
        interp.append(f"- Unconditional crash risk is {uncond_cr}/{uncond_obs} = {uncond_p:.1f}%, while strict 100% run-up months show {strict_cr}/{strict_obs} = {strict_p:.1f}%.")

if {'Comparison', 'Difference (percentage points)'}.issubset(crash_test.columns):
    diffs = crash_test.dropna(subset=['Difference (percentage points)'])
    if not diffs.empty:
        d = float(diffs['Difference (percentage points)'].iloc[0])
        interp.append(f"- The estimated difference is about {d:.1f} percentage points in the current output.")

if {'Comparison', 'z statistic', 'p-value'}.issubset(crash_test.columns):
    tested = crash_test.dropna(subset=['z statistic', 'p-value'])
    if len(tested) > 0:
        tests_text = '; '.join(
            f"{row['Comparison']}: z={row['z statistic']:.3f}, p={row['p-value']:.6f}" for _, row in tested.iterrows()
        )
        interp.append(f"- Formal tests are statistically significant ({tests_text}).")

interp.append('- Despite formal significance, inference should be tempered because the strict 100% group has only five observations and may contain dependence/overlap concerns across months and industries.')

display(Markdown('\n'.join(interp)))


## 15. Comparison with the U.S. evidence

The comparison should be organized around patterns rather than a mechanical equality of numerical estimates. The U.S. study contains many more industries and episodes, while the German analysis contains a smaller country-specific sample.

The relevant comparison questions are:

- Does Germany also contain large industry run-ups followed by substantial drawdowns?
- Is the crash probability after a strict 100% run-up higher than the broad unconditional rate?
- Do event-time paths separate between crash and non-crash episodes?
- Does controlling for pre-run-up volatility reduce the run-up relationship?

Evidence that points in the same direction as the U.S. results should be described as consistency with the broader pattern. Differences should be discussed in light of Germany's smaller sample, industry composition, currency history, and sample period.

## 16. Integrity checks and reproducibility

The final exhibit files are the authoritative reporting outputs. Supporting Stata files remain in the project folder so that the workflow can be inspected and rerun. This notebook does not overwrite project files.

In [ ]:
supporting_files = [
    'germany_prepared.dta',
    'germany_currency_clean.dta',
    'germany_returns_raw.dta',
    'germany_daily_clean.dta',
    'germany_monthly_security.dta',
    'germany_industry_monthly.dta',
    'germany_runup_monthly.dta',
    'germany_runup_outcomes.dta',
    'germany_industry_volatility.dta',
    'germany_gsy_analysis.dta',
]
for filename in supporting_files:
    print(f'{filename}: {"FOUND" if (PROJECT_DIR / filename).exists() else "MISSING"}')

print('Diagnostic presentation complete.')


## 17. Limitations

The main limitations are the small number of German run-up episodes, the use of one WRDS source, the fixed DEM/EUR conversion, the absence of an external risk-free series, possible differences in industry composition relative to the U.S. analysis, and the descriptive nature of the event-time figure.

The formal tests and regressions should therefore not be presented as definitive causal evidence. They provide structured evidence about whether the German data display the same broad run-up and crash pattern.

## 18. Final synthesis

This section consolidates the displayed exhibits into a direct answer to the research question using the currently generated German outputs.


In [ ]:
summary_table1 = table1.copy() if 'table1' in globals() else read_excel('germany_table1_paper_style.xlsx')
summary_detail = read_excel('germany_crash_results.xlsx') if (PROJECT_DIR / 'germany_crash_results.xlsx').exists() else None
summary_reg = regressions.copy() if 'regressions' in globals() else read_excel('germany_footnote15_regressions.xlsx')
summary_test = crash_test.copy() if 'crash_test' in globals() else read_excel('germany_crash_test.xlsx')

lines = []
lines.append('### Integrated interpretation')

if summary_detail is not None and {'Pick-up threshold (%)', 'Number of run-ups identified', 'Raw crashes (%)'}.issubset(summary_detail.columns):
    row100 = summary_detail.loc[summary_detail['Pick-up threshold (%)'] == 100]
    if not row100.empty:
        n100 = int(row100['Number of run-ups identified'].iloc[0])
        c100 = float(row100['Raw crashes (%)'].iloc[0])
        lines.append(f"- **Research question answer (Germany):** large run-ups are followed by materially higher crash risk in the strict sample (100% threshold: {n100} episodes, raw crash rate {c100:.1f}%).")

if {'Comparison', 'Crash probability (%)'}.issubset(summary_test.columns):
    idx = summary_test.set_index('Comparison')
    if 'Unconditional' in idx.index and '100% run-up' in idx.index:
        lines.append(f"- Unconditional crash probability is {float(idx.loc['Unconditional', 'Crash probability (%)']):.1f}% versus {float(idx.loc['100% run-up', 'Crash probability (%)']):.1f}% for strict run-up observations, with significant z-tests in the exhibit.")

if {'Return type', 'Volatility control', 'Run-up coefficient'}.issubset(summary_reg.columns):
    no = summary_reg[summary_reg['Volatility control'].astype(str).str.lower() == 'no']['Run-up coefficient']
    yes = summary_reg[summary_reg['Volatility control'].astype(str).str.lower() == 'yes']['Run-up coefficient']
    if not no.empty and not yes.empty:
        lines.append(f"- Adding volatility controls lowers average run-up coefficients ({no.mean():.3f} to {yes.mean():.3f}), so pre-run-up risk conditions explain part—but not all—of the crash association.")

lines.append('- **Germany versus U.S. framing:** the direction of evidence is broadly similar (run-ups linked to elevated crash risk), but this notebook does not claim identical replication magnitudes because Germany has far fewer strict episodes and a different market structure.')
lines.append('- **Limitations:** very small strict-event sample, potential dependence/overlap in monthly panels, unavailable risk-free fields in the current setup, and descriptive event-time evidence that should not be read as causal proof.')

display(Markdown('\n'.join(lines)))
